In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.6 MB/s eta 0:00:00


In [10]:
# =========================================================
# 정면 상추(개별 crop 이미지) 지표 추출 - STEP1
# - YOLOv8-seg로 마스크 추론 → 면적/shape/QC + 정면 전용 지표
# - scale_map(CSV)로 px→cm 환산
# - 재귀탐색 / chunk / batch / 병렬 로딩 / 진행률(ETA) 포함
# =========================================================
# 사용법:
# 1) 아래 CFG 경로 3개만 수정
# 2) 런타임에 ultralytics/opencv/pandas 설치되어 있어야 함
# 3) 맨 아래 main() 실행
# =========================================================

import os
import re
import time
from glob import glob
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO

# ============================
# CFG: 경로/파라미터 (여기만 수정)
# ============================
FRONT_CROP_FOLDER = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/lettuce/lettuce_crops_out"   # 정면 상추 crop 이미지들이 모여있는 폴더(재귀 탐색)
SEG_MODEL_PATH    = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/model/results/runs/bed_seg_v1/weights/best.pt"  # 정면 상추 seg 모델(.pt)
SCALE_CSV         = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/lettuce/260128_daily_scale_1st.csv"  # scale_map CSV (없으면 ""로)
OUT_xlsx          = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/lettuce/260128_front_step1_features.xlsx"  # 결과 저장 경로

# 처리 파라미터
BATCH_SIZE  = 24
CHUNK_SIZE  = 3000
N_WORKERS   = 4

# YOLO 추론 파라미터
PRED_CONF = 0.05
PRED_IOU  = 0.7

# 추가지표 파라미터
BOTTOM_BAND_RATIO = 0.12   # 하단 flatness 계산 시, bbox_h의 하단 몇 %를 볼지
CORE_Q            = 90     # core prominence: distance transform 상위 CORE_Q 분위수 이상 비율


In [11]:

# ============================
# 유틸
# ============================
IMG_PATTERNS = ["**/*.jpg", "**/*.JPG", "**/*.jpeg", "**/*.JPEG", "**/*.png", "**/*.PNG"]


def now_hms():
    return time.strftime("%H:%M:%S")


def brightness_mean(img_bgr: np.ndarray) -> float:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return float(gray.mean())


def blur_score(img_bgr: np.ndarray) -> float:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def get_main_contour(mask_u8: np.ndarray):
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    return max(contours, key=cv2.contourArea)


def calc_shape_descriptors(contour):
    """윗면 코드와 동일한 형태 지표 (정면에도 그대로 적용)."""
    out = {
        "bbox_w": np.nan,
        "bbox_h": np.nan,
        "perimeter_px": np.nan,
        "circularity": np.nan,
        "solidity": np.nan,
        "concavity": np.nan,
        "curvature": np.nan,
        "roughness": np.nan,
    }

    if contour is None:
        return out

    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    x, y, w, h = cv2.boundingRect(contour)

    out["bbox_w"] = float(w)
    out["bbox_h"] = float(h)
    out["perimeter_px"] = float(perimeter)

    if perimeter > 0:
        out["circularity"] = float(4 * np.pi * area / (perimeter ** 2))

    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    if hull_area > 0:
        sol = area / hull_area
        out["solidity"] = float(sol)
        out["concavity"] = float(1 - sol)

    pts = contour.squeeze()
    if len(pts) > 10:
        diffs = np.diff(pts, axis=0)
        angles = np.arctan2(diffs[:, 1], diffs[:, 0])
        curv = np.abs(np.diff(angles))
        out["curvature"] = float(np.mean(curv))
        out["roughness"] = float(np.std(curv))

    return out


def parse_ids_from_filename(path: str):
    """파일명에서 base_key(원본베드ID)와 lettuce_id(t2,b3 등) 추출.

    기대 형태 예:
      bed03_20251226_080317_cam2_t2.jpg

    - base_key: bed03_20251226_080317_cam2
    - lettuce_id: t2
    """
    stem = os.path.splitext(os.path.basename(path))[0]

    m = re.match(r"^(.*)_(t\d+|b\d+)$", stem)
    if m:
        base_key = m.group(1)
        lettuce_id = m.group(2)
        return base_key, lettuce_id

    # fallback: cam까지를 base_key로
    m2 = re.match(r"^(.*_cam\d+)(?:_.*)?$", stem)
    base_key = m2.group(1) if m2 else stem
    lettuce_id = "unknown"
    return base_key, lettuce_id


In [12]:
# ============================
# scale_map 로드
# ============================

def load_scale_map(scale_csv: str):
    if not isinstance(scale_csv, str) or len(scale_csv) == 0 or not os.path.exists(scale_csv):
        print("[INFO] SCALE_CSV 미사용: cm 단위는 NaN으로 저장됩니다.")
        return None

    df = pd.read_csv(scale_csv)

    # 키 컬럼 보정: parent_name 또는 image_name → parent_name
    if "parent_name" not in df.columns:
        if "image_name" in df.columns:
            df["parent_name"] = df["image_name"].astype(str).apply(lambda x: os.path.splitext(os.path.basename(x))[0])
        else:
            raise ValueError("SCALE_CSV에 parent_name 또는 image_name 컬럼이 필요합니다.")

    smap = df.set_index("parent_name").to_dict(orient="index")
    print(f"[INFO] scale_map 로드 완료: {len(smap)} rows")
    return smap


def scale_lookup(scale_map, base_key: str):
    """base_key로 scale_map 매칭 (정면 scale_map.csv 스키마 반영)

    사용자 scale_map 컬럼 예:
      path, image_name, date, cyl_ok, cyl_diam_px, cm_per_px_today, qc_flag, method

    반환:
      (mm_per_px, px_per_mm_x, px_per_mm_y, cm_per_px_x, cm_per_px_y, cyl_ok, cyl_diam_px)

    규칙:
      - cm_per_px_today가 있으면 최우선 사용
      - px_per_mm_x/y가 scale_map에 없어도, cm_per_px_today로부터 “일관되게” 계산해 채움
        * 1px = cm_per_px_today (cm)
        * 1px = cm_per_px_today*10 (mm) => mm_per_px
        * 1mm = 1/(mm_per_px) (px)     => px_per_mm
      - legacy(mm_per_px / px_per_mm_x,y)도 있으면 대응
    """
    if scale_map is None or base_key not in scale_map:
        return (np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan)

    info = scale_map[base_key]

    mm_per_px = np.nan
    pxmm_x = np.nan
    pxmm_y = np.nan
    cmppx_x = np.nan
    cmppx_y = np.nan
    cyl_ok = np.nan
    cyl_diam_px = np.nan

    if "cyl_ok" in info and pd.notna(info["cyl_ok"]):
        cyl_ok = float(info["cyl_ok"])
    if "cyl_diam_px" in info and pd.notna(info["cyl_diam_px"]):
        cyl_diam_px = float(info["cyl_diam_px"])

    # (1) 정면 scale_map 핵심: cm_per_px_today
    if "cm_per_px_today" in info and pd.notna(info["cm_per_px_today"]):
        cmppx_x = float(info["cm_per_px_today"])
        cmppx_y = float(info["cm_per_px_today"])

        mm_per_px = cmppx_x * 10.0  # 1px = ? mm
        if mm_per_px > 0:
            pxmm_x = 1.0 / mm_per_px  # 1mm = ? px
            pxmm_y = 1.0 / mm_per_px

        return (mm_per_px, pxmm_x, pxmm_y, cmppx_x, cmppx_y, cyl_ok, cyl_diam_px)

    # (2) 레거시: px_per_mm_x/y
    if ("px_per_mm_x" in info and "px_per_mm_y" in info and
        pd.notna(info["px_per_mm_x"]) and pd.notna(info["px_per_mm_y"])):
        pxmm_x = float(info["px_per_mm_x"])
        pxmm_y = float(info["px_per_mm_y"])
        cmppx_x = (1.0 / pxmm_x) / 10.0
        cmppx_y = (1.0 / pxmm_y) / 10.0
        mm_per_px = ((cmppx_x + cmppx_y) / 2.0) * 10.0
        return (mm_per_px, pxmm_x, pxmm_y, cmppx_x, cmppx_y, cyl_ok, cyl_diam_px)

    # (3) 레거시: mm_per_px
    if "mm_per_px" in info and pd.notna(info["mm_per_px"]):
        mm_per_px = float(info["mm_per_px"])
        cmppx_x = mm_per_px / 10.0
        cmppx_y = mm_per_px / 10.0
        if mm_per_px > 0:
            pxmm_x = 1.0 / mm_per_px
            pxmm_y = 1.0 / mm_per_px
        return (mm_per_px, pxmm_x, pxmm_y, cmppx_x, cmppx_y, cyl_ok, cyl_diam_px)

    return (mm_per_px, pxmm_x, pxmm_y, cmppx_x, cmppx_y, cyl_ok, cyl_diam_px)



# ============================
# 정면 전용 추가 지표
# ============================

def compute_bottom_flatness(mask_u8: np.ndarray, contour, band_ratio: float = 0.12):
    """하단 flatness(접지) proxy.

    - bbox 기준 하단 band에서 mask의 최대 x-span / bbox_w
    - 값이 클수록 하단에서 "평평하게 넓게" 붙어있는 경향
    """
    if contour is None:
        return np.nan

    x, y, w, h = cv2.boundingRect(contour)
    if w <= 0 or h <= 0:
        return np.nan

    band = max(3, int(h * band_ratio))
    y0 = min(mask_u8.shape[0] - 1, y + h - 1)
    y1 = max(0, y0 - band)

    region = mask_u8[y1:y0 + 1, x:x + w]
    if region.size == 0:
        return np.nan

    # 행별 x-span
    max_span = 0
    for r in region:
        xs = np.where(r > 0)[0]
        if xs.size >= 2:
            span = int(xs.max() - xs.min() + 1)
            if span > max_span:
                max_span = span
        elif xs.size == 1:
            max_span = max(max_span, 1)

    return float(max_span / w)


def compute_core_prominence(mask_u8: np.ndarray, q: int = 90):
    """결구 코어 prominence proxy.

    distance transform을 이용해서 "두께"가 큰 픽셀 비율을 계산.
    - mask 내부 dist 값의 상위 q 분위수 이상인 픽셀 비율

    값이 클수록 중심부가 두껍게(둥글게) 모여있는 경향.
    """
    if mask_u8 is None or mask_u8.max() == 0:
        return np.nan

    # distanceTransform은 0/255 형태를 권장
    m255 = (mask_u8 > 0).astype(np.uint8) * 255
    dist = cv2.distanceTransform(m255, cv2.DIST_L2, 5)

    vals = dist[dist > 0]
    if vals.size == 0:
        return np.nan

    thr = np.percentile(vals, q)
    core_ratio = float((dist >= thr).sum() / (dist > 0).sum())
    return core_ratio



In [13]:

# ============================
# 메인 추출
# ============================

def collect_image_paths(root: str):
    paths = []
    for pat in IMG_PATTERNS:
        paths.extend(glob(os.path.join(root, pat), recursive=True))
    paths = sorted(set(paths))
    return paths


def main():
    print(f"[{now_hms()}] [INFO] 시작")

    # 1) 입력 수집
    image_paths = collect_image_paths(FRONT_CROP_FOLDER)
    print(f"[INFO] 총 정면 crop 이미지 개수: {len(image_paths)}")
    if len(image_paths) == 0:
        raise RuntimeError("FRONT_CROP_FOLDER에 이미지가 없습니다.")

    # 2) scale_map
    scale_map = load_scale_map(SCALE_CSV)

    # 3) 모델 로드
    model = YOLO(SEG_MODEL_PATH)
    print("[INFO] model task:", getattr(model, "task", "unknown"))

    rows = []
    t0 = time.time()

    n_chunks = (len(image_paths) - 1) // CHUNK_SIZE + 1
    print(f"[INFO] chunk 분할: {n_chunks}개 (chunk 당 {CHUNK_SIZE}장)")

    # 디버그 카운터
    cnt_read_ok = 0
    cnt_pred = 0
    cnt_with_masks = 0

    # ---------- chunk loop ----------
    for c_start in range(0, len(image_paths), CHUNK_SIZE):
        chunk_paths = image_paths[c_start:c_start + CHUNK_SIZE]
        chunk_idx = c_start // CHUNK_SIZE + 1

        pbar = tqdm(
            range(0, len(chunk_paths), BATCH_SIZE),
            desc=f"Chunk {chunk_idx}/{n_chunks}",
            leave=True,
        )

        for i in pbar:
            batch_paths = chunk_paths[i:i + BATCH_SIZE]

            # 병렬 로딩
            with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
                imgs = list(ex.map(cv2.imread, batch_paths))

            valid = [(p, im) for p, im in zip(batch_paths, imgs) if im is not None]
            cnt_read_ok += len(valid)
            if not valid:
                continue

            paths, imgs = zip(*valid)

            # YOLO seg 추론
            results = model(list(imgs), verbose=False, conf=PRED_CONF, iou=PRED_IOU)
            cnt_pred += len(results)

            # ETA 표시(대략)
            elapsed = time.time() - t0
            done = (c_start + i + len(batch_paths))
            frac = min(1.0, done / max(1, len(image_paths)))
            if frac > 0:
                eta = elapsed * (1 - frac) / frac
                pbar.set_postfix_str(f"elapsed={elapsed/60:.1f}m eta={eta/60:.1f}m")

            for img_path, img, result in zip(paths, imgs, results):
                H, W = img.shape[:2]
                filename = os.path.basename(img_path)

                # key 파싱
                base_key, lettuce_id = parse_ids_from_filename(img_path)

                # QC
                b_mean = brightness_mean(img)
                b_blur = blur_score(img)

                masks = result.masks
                boxes = result.boxes

                if masks is None or boxes is None or len(masks) == 0:
                    # 검출 실패도 기록은 남겨두는 편이 디버깅에 좋음
                    rows.append({
                        "image_path": img_path,
                        "base_key": base_key,
                        "lettuce_id": lettuce_id,
                        "filename": filename,
                        "n_instances": 0,
                        "conf": np.nan,
                        "brightness_mean": b_mean,
                        "blur_score": b_blur,
                        "area_px": 0,
                        "area_cm2": np.nan,
                        "px_per_mm_x": np.nan,
                        "px_per_mm_y": np.nan,
                        "mm_per_px": np.nan,
                        "cyl_ok": np.nan,
                        "cyl_diam_px": np.nan,
                        "front_height_cm": np.nan,
                        "area_front": 0,
                        "aspect_ratio": np.nan,
                        "bottom_flatness": np.nan,
                        "core_prominence": np.nan,
                        "bbox_w": np.nan,
                        "bbox_h": np.nan,
                        "perimeter_px": np.nan,
                        "circularity": np.nan,
                        "solidity": np.nan,
                        "concavity": np.nan,
                        "curvature": np.nan,
                        "roughness": np.nan,
                    })
                    continue

                cnt_with_masks += 1

                # 인스턴스 여러 개면: score = conf*area로 대표 1개 선택(정면 crop은 보통 1개)
                conf_list = boxes.conf.cpu().numpy().astype(float)
                masks_np = masks.data.cpu().numpy().astype(np.uint8)

                areas = []
                resized_masks = []
                for k in range(len(masks_np)):
                    mk = cv2.resize(masks_np[k], (W, H), interpolation=cv2.INTER_NEAREST)
                    resized_masks.append(mk)
                    areas.append(cv2.countNonZero(mk))
                areas = np.array(areas, dtype=float)
                scores = conf_list * areas
                best_idx = int(np.argmax(scores))

                mask_u8 = resized_masks[best_idx]
                area_px = int(areas[best_idx])

                contour = get_main_contour(mask_u8)
                shape = calc_shape_descriptors(contour)

                # scale
                mm_per_px, pxmm_x, pxmm_y, cmppx_x, cmppx_y, cyl_ok, cyl_diam_px = scale_lookup(scale_map, base_key)

                # area_cm2 (가능하면 방향별 스케일 사용)
                if np.isfinite(cmppx_x) and np.isfinite(cmppx_y):
                    area_cm2 = float(area_px * (cmppx_x * cmppx_y))
                else:
                    area_cm2 = np.nan

                # 정면 추가 지표
                bbox_w = shape.get("bbox_w", np.nan)
                bbox_h = shape.get("bbox_h", np.nan)

                # front_height_cm: bbox_h * cm_per_px_y
                front_height_cm = float(bbox_h * cmppx_y) if (np.isfinite(bbox_h) and np.isfinite(cmppx_y)) else np.nan

                area_front = area_px  # 정면 투영면적(px)
                aspect_ratio = float(bbox_w / bbox_h) if (np.isfinite(bbox_w) and np.isfinite(bbox_h) and bbox_h > 0) else np.nan

                bottom_flat = compute_bottom_flatness(mask_u8, contour, band_ratio=BOTTOM_BAND_RATIO)
                core_prom = compute_core_prominence(mask_u8, q=CORE_Q)

                rows.append({
                    "image_path": img_path,
                    "base_key": base_key,
                    "lettuce_id": lettuce_id,
                    "filename": filename,
                    "n_instances": int(len(masks_np)),
                    "best_instance": best_idx,
                    "conf": float(conf_list[best_idx]),
                    "brightness_mean": b_mean,
                    "blur_score": b_blur,

                    # 기본 지표
                    "area_px": area_px,
                    "area_cm2": area_cm2,
                    "px_per_mm_x": pxmm_x,
                    "px_per_mm_y": pxmm_y,
                    "mm_per_px": mm_per_px,
                    "cyl_ok": cyl_ok,
                    "cyl_diam_px": cyl_diam_px,


                    # shape
                    "perimeter_px": shape.get("perimeter_px", np.nan),
                    "circularity": shape.get("circularity", np.nan),
                    "solidity": shape.get("solidity", np.nan),
                    "concavity": shape.get("concavity", np.nan),
                    "curvature": shape.get("curvature", np.nan),
                    "roughness": shape.get("roughness", np.nan),
                    "bbox_w": bbox_w,
                    "bbox_h": bbox_h,

                    # 정면 추가 지표
                    "front_height_cm": front_height_cm,
                    "area_front": area_front,
                    "aspect_ratio": aspect_ratio,
                    "bottom_flatness": bottom_flat,
                    "core_prominence": core_prom,
                })

    # 저장
    df = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(OUT_xlsx), exist_ok=True)
    df.to_excel(OUT_xlsx, index=False, engine="openpyxl")


    elapsed = time.time() - t0
    print(f"[{now_hms()}] [DONE] saved: {OUT_xlsx}")
    print(f"[INFO] elapsed: {elapsed/60:.1f} min")
    print(f"[INFO] read_ok={cnt_read_ok} pred_batches={cnt_pred} with_masks={cnt_with_masks}")
    print(df.head(3))



In [14]:
main()


[14:24:44] [INFO] 시작
[INFO] 총 정면 crop 이미지 개수: 8768
[INFO] scale_map 로드 완료: 2192 rows
[INFO] model task: segment
[INFO] chunk 분할: 3개 (chunk 당 3000장)


Chunk 3/3: 100%|██████████| 116/116 [05:43<00:00,  2.96s/it, elapsed=18.4m eta=0.0m]


[14:43:27] [DONE] saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/lettuce/260128_front_step1_features.xlsx
[INFO] elapsed: 18.5 min
[INFO] read_ok=8768 pred_batches=8768 with_masks=7163
                                          image_path  \
0  /content/drive/Othercomputers/내 컴퓨터/새 포...   
1  /content/drive/Othercomputers/내 컴퓨터/새 포...   
2  /content/drive/Othercomputers/내 컴퓨터/새 포...   

                     base_key lettuce_id                           filename  \
0  bed00_20251204_093629_cam2         b1  bed00_20251204_093629_cam2_b1.jpg   
1  bed00_20251204_093629_cam2         b2  bed00_20251204_093629_cam2_b2.jpg   
2  bed00_20251204_093629_cam2         t1  bed00_20251204_093629_cam2_t1.jpg   

   n_instances      conf  brightness_mean  blur_score  area_px    area_cm2  \
0            0       NaN        63.736762   80.157397        0         NaN   
1            0       NaN        